In [0]:
events = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")
)

In [0]:
dbutils.widgets.text("layer", "silver")
layer = dbutils.widgets.get("layer")

print("Running layer:", layer)

In [0]:
from pyspark.sql import functions as F

def create_user_features(events_df):
    
    features_df = events_df.groupBy("user_id").agg(
        F.count("*").alias("total_events"),
        F.count(F.when(F.col("event_type") == "purchase", 1)).alias("purchases"),
        F.sum("price").alias("total_spent"),
        F.avg("price").alias("avg_price")
    )
    
    return features_df

In [0]:
features_df = create_user_features(events)
features_df.display()

In [0]:
table_name = f"{layer}_user_features"

features_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"Data saved to table: {table_name}")